# 04 — App Demo (Streamlit)

Este notebook explica cómo funciona la app Streamlit y cómo lanzarla.

La app en `app/app.py` junta todo lo que hemos construido en una interfaz visual:
- Chat con los papers
- Selector de estrategia de retrieval (Naive, HyDE, RAG-Fusion, CrossEncoder)
- Muestra los chunks recuperados como fuentes

**Prerequisito**: haber ejecutado el notebook 01 (vector store creado en `chroma_db/`)

## Arquitectura de la app

```
Usuario escribe pregunta
        ↓
Selector de estrategia (sidebar)
        ↓
  ┌─────────────────────────────────┐
  │  Naive  │ HyDE │ Fusion │ Cross │
  └─────────────────────────────────┘
        ↓
ChromaDB → k chunks relevantes
        ↓
Groq (Llama 3.3 70B) → respuesta
        ↓
Chat UI + fuentes expandibles
```

## Cómo lanzar la app

Desde la raíz del proyecto:

In [1]:
# Muestra el comando para lanzar la app (ejecútalo en terminal, no aquí)
import subprocess, sys
from pathlib import Path

ROOT = Path().resolve().parent
print('Lanza la app con este comando desde la raíz del proyecto:')
print(f'\n  streamlit run app/app.py\n')
print('Se abrirá automáticamente en: http://localhost:8501')

Lanza la app con este comando desde la raíz del proyecto:

  streamlit run app/app.py

Se abrirá automáticamente en: http://localhost:8501


## Verificación previa — ¿el vector store está listo?

In [2]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

chroma_path = ROOT / 'chroma_db'
if chroma_path.exists() and any(chroma_path.iterdir()):
    print('✅ Vector store encontrado — la app puede arrancar')
else:
    print('❌ Vector store no encontrado.')
    print('   Ejecuta primero el notebook 01_naive_rag.ipynb completo.')

✅ Vector store encontrado — la app puede arrancar


## Demo rápida en el propio notebook

Si no quieres abrir la app, puedes hacer preguntas directamente aquí.
Cambia `STRATEGY` para probar distintas técnicas.

In [ ]:
from src.ingestion.embedder import get_embeddings
from src.retrieval.vectorstore import load_vectorstore
from src.generation.chain import build_rag_chain, ask
from src.retrieval.strategies import retrieve_hyde, retrieve_rag_fusion, retrieve_with_reranking
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from src.generation.chain import format_docs
import os

embeddings = get_embeddings()
vectorstore = load_vectorstore(embeddings)

RAG_PROMPT = ChatPromptTemplate.from_template("""
Responde basándote ÚNICAMENTE en el contexto.
Contexto: {context}
Pregunta: {question}
Respuesta:""")

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, api_key=os.getenv('GROQ_API_KEY'))
print('Listo')

In [ ]:
# ── Cambia estos valores y re-ejecuta la celda ──
QUESTION = 'What is the difference between RAG and fine-tuning a language model?'
STRATEGY = 'CrossEncoder'  # opciones: 'Naive', 'HyDE', 'RAG-Fusion', 'CrossEncoder'
K = 4
# ────────────────────────────────────────────────

if STRATEGY == 'Naive':
    docs = vectorstore.similarity_search(QUESTION, k=K)
elif STRATEGY == 'HyDE':
    docs = retrieve_hyde(vectorstore, QUESTION, k=K)
elif STRATEGY == 'RAG-Fusion':
    docs = retrieve_rag_fusion(vectorstore, QUESTION, k=K)
elif STRATEGY == 'CrossEncoder':
    docs = retrieve_with_reranking(vectorstore, QUESTION, k_initial=K*3, k_final=K)

context = format_docs(docs)
answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({'context': context, 'question': QUESTION})

print(f'Estrategia: {STRATEGY}')
print(f'Pregunta:   {QUESTION}')
print(f'\nRespuesta:\n{answer}')
print(f'\nFuentes: {[d.metadata["source"] for d in docs]}')

## Lanzar la app Streamlit desde aquí

In [ ]:
# Lanza la app en segundo plano (abre http://localhost:8501)
import subprocess
proc = subprocess.Popen(
    ['streamlit', 'run', str(ROOT / 'app' / 'app.py'), '--server.headless=true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f'App lanzada (PID {proc.pid})')
print('Abre: http://localhost:8501')

## Resumen del proyecto completo

Has construido un RAG pipeline de producción desde cero:

| Fase | Qué aprendiste |
|------|----------------|
| **01 Naive RAG** | Chunking, embeddings, vector search, generación con LLM |
| **02 Advanced** | HyDE, RAG-Fusion, CrossEncoder reranking |
| **03 Evaluation** | Medir con RAGAS: faithfulness, relevancy, precision, recall |
| **04 App** | Streamlit: interfaz de chat con selector de estrategia |

**Stack 100% gratuito:**
- BGE-M3 (embeddings locales)
- ChromaDB (vector store local)
- Groq / Llama 3.3 70B (LLM gratuito)
- RAGAS (evaluación open source)
- Streamlit (demo visual)

---
*RAG pipeline — built from scratch · NikitoGlez*